In [3]:
import os
import cv2
import numpy as np


class TableStructureDetector:

    def __init__(self):

        # CLAHE settings
        self.clip_limit = 2.0
        self.tile_grid_size = (8, 8)

        # Threshold settings
        self.adaptive_block_size = 15
        self.adaptive_c = 10

        # Morphology settings
        self.horizontal_kernel_size = (60, 1)
        self.vertical_kernel_size = (1, 60)
        self.close_kernel_size = (10, 10)

    def process(self, image_path, image_id):

        # Check file exists
        if not os.path.exists(image_path):
            print(f"{image_path} Path not found")
            return None

        # Read image
        img = cv2.imread(image_path)

        if img is None:
            print(f"{image_path} Image not found")
            return None

        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # CLAHE
        clahe = cv2.createCLAHE(
            clipLimit=self.clip_limit,
            tileGridSize=self.tile_grid_size
        )

        gray_clahe = clahe.apply(gray)

        # Adaptive threshold
        thresh = cv2.adaptiveThreshold(
            gray_clahe,
            255,
            cv2.ADAPTIVE_THRESH_MEAN_C,
            cv2.THRESH_BINARY_INV,
            self.adaptive_block_size,
            self.adaptive_c
        )

        # Horizontal line detection
        horizontal_kernel = cv2.getStructuringElement(
            cv2.MORPH_RECT,
            self.horizontal_kernel_size
        )

        horizontal = cv2.morphologyEx(
            thresh,
            cv2.MORPH_OPEN,
            horizontal_kernel
        )

        horizontal = cv2.dilate(
            horizontal,
            np.ones((2, 2), np.uint8),
            iterations=2
        )

        # Vertical line detection
        vertical_kernel = cv2.getStructuringElement(
            cv2.MORPH_RECT,
            self.vertical_kernel_size
        )

        vertical = cv2.morphologyEx(
            thresh,
            cv2.MORPH_OPEN,
            vertical_kernel
        )

        vertical = cv2.dilate(
            vertical,
            np.ones((2, 2), np.uint8),
            iterations=2
        )

        # Combine horizontal + vertical
        boxes = cv2.add(horizontal, vertical)

        # Close gaps
        kernel = np.ones(self.close_kernel_size, np.uint8)

        boxes = cv2.morphologyEx(
            boxes,
            cv2.MORPH_CLOSE,
            kernel,
            iterations=2
        )

        # Add border
        h, w = img.shape[:2]

        border = cv2.rectangle(
            boxes.copy(),
            (0, 0),
            (w, h),
            (255, 255, 255),
            3
        )

        # Find contours
        # contours, hierarchy = cv2.findContours(
        #     border,
        #     cv2.RETR_TREE,
        #     cv2.CHAIN_APPROX_SIMPLE
        # )

        return img

In [4]:
from paddleocr import PaddleOCR 
import paddle
import cv2
import numpy as np
import os
from llama_cpp import Llama
import json

In [6]:
i="4"
image_path = f"D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/{i}.jpg"
image_path

'D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/4.jpg'

In [7]:
img = cv2.imread(image_path)

# extracting the table from the img 

In [8]:
# converting img to gray
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
# cv2.imwrite(f'./output/opencv/gray{i}.jpeg', gray)

In [9]:
# splits the image and clahe 
# C – Contrast
# L – Limited
# A – Adaptive
# H – Histogram
# E – Equalization
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
gray_clahe = clahe.apply(gray)    
# cv2.imwrite(f'./output/opencv/gray_clahe{i}.jpeg', gray_clahe)

In [10]:
# Better threshold (adaptive handles uneven lighting)
thresh = cv2.adaptiveThreshold(
    gray_clahe, 255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY_INV,
    15, 10
)

In [11]:
# --- Horizontal lines ---
horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (60,1))
horizontal = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel)
print(horizontal_kernel)
print(horizontal)
# cv2.imwrite(f'./output/opencv/horizontal{i}.jpeg', horizontal)

[[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
  1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]]
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [12]:
# connect broken horizontal lines
horizontal = cv2.dilate(horizontal, np.ones((2,2),np.uint8), iterations=2)
print(horizontal)
cv2.imwrite(f'./output/opencv/horizontal_dilated{i}.jpeg', horizontal)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


True

In [ ]:
# --- Vertical lines ---
vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1,60))
vertical = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vertical_kernel)
# cv2.imwrite(f'./output/opencv/vertical{i}.jpeg', vertical)
 

In [ ]:
# connect broken vertical lines
vertical = cv2.dilate(vertical, np.ones((2,2),np.uint8), iterations=2)
cv2.imwrite(f'./output/opencv/vertical_dilate{i}.jpeg', vertical)



In [ ]:
# Combine
boxes = cv2.add(horizontal, vertical)
# cv2.imwrite(f'./output/opencv/boxes_combine{i}.jpeg', boxes)

In [ ]:
# Final closing to join gaps
kernel = np.ones((10,10), np.uint8)
boxes = cv2.morphologyEx(boxes, cv2.MORPH_CLOSE, kernel, iterations=2)
cv2.imwrite(f'./output/opencv/boxes_fill_gap{i}.jpeg', boxes)  
# cv2.imwrite(f'./output/opencv/new/boxes_fill_gap{i}.jpeg', boxes)  

In [ ]:
# adding border
h, w = img.shape[:2]
border=cv2.rectangle(boxes, (0,0), (w,h), (255,255,255), 3)
cv2.imwrite(f'./output/opencv/bordered{i}.jpeg', border)
# cv2.imwrite(f'./output/opencv/new/bordered{i}.jpeg', border)

In [ ]:
# Find contours
contours, _ = cv2.findContours(border, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
output = img.copy()
cv2.imwrite(f'./output/opencv/output{i}.jpeg', output)
# cv2.imwrite(f'./output/opencv/new/output{i}.jpeg', output)

In [ ]:
if (img is None) :
    print(f"{image_path} this is incorrect no img found")
else:
   
    # converting img to gray
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # cv2.imwrite(f'./output/opencv/gray{i}.jpeg', gray)

    # splits the image and clahe 
    # C – Contrast
    # L – Limited
    # A – Adaptive
    # H – Histogram
    # E – Equalization
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    gray_clahe = clahe.apply(gray)    
    # cv2.imwrite(f'./output/opencv/gray_clahe{i}.jpeg', gray_clahe)
    
    # Better threshold (adaptive handles uneven lighting)
    thresh = cv2.adaptiveThreshold(
        gray_clahe, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        15, 10
    )
     
    
    # --- Horizontal lines ---
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (60,1))
    horizontal = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel)
    # cv2.imwrite(f'./output/opencv/horizontal{i}.jpeg', horizontal)
    
    # connect broken horizontal lines
    horizontal = cv2.dilate(horizontal, np.ones((2,2),np.uint8), iterations=2)
    # cv2.imwrite(f'./output/opencv/horizontal_dilated{i}.jpeg', horizontal)
    
    # --- Vertical lines ---
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1,60))
    vertical = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vertical_kernel)
    # cv2.imwrite(f'./output/opencv/vertical{i}.jpeg', vertical)
     
    # connect broken vertical lines
    vertical = cv2.dilate(vertical, np.ones((2,2),np.uint8), iterations=2)
    # cv2.imwrite(f'./output/opencv/vertical_dilate{i}.jpeg', vertical)
    
    
    # Combine
    boxes = cv2.add(horizontal, vertical)
    # cv2.imwrite(f'./output/opencv/boxes_combine{i}.jpeg', boxes)
    
    # Final closing to join gaps
    kernel = np.ones((10,10), np.uint8)
    boxes = cv2.morphologyEx(boxes, cv2.MORPH_CLOSE, kernel, iterations=2)
    cv2.imwrite(f'./output/opencv/boxes_fill_gap{i}.jpeg', boxes)  
    # cv2.imwrite(f'./output/opencv/new/boxes_fill_gap{i}.jpeg', boxes)  

    # kernel = np.ones((2,2), np.uint8)
    # erode = cv2.erode(boxes, kernel, iterations=2)
    # cv2.imwrite(f'./output/opencv/erode{i}.jpeg', erode)
    
    # adding border
    h, w = img.shape[:2]
    border=cv2.rectangle(boxes, (0,0), (w,h), (255,255,255), 3)
    cv2.imwrite(f'./output/opencv/bordered{i}.jpeg', border)
    # cv2.imwrite(f'./output/opencv/new/bordered{i}.jpeg', border)
    
    # Find contours
    contours, _ = cv2.findContours(border, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    output = img.copy()
    cv2.imwrite(f'./output/opencv/output{i}.jpeg', output)
    # cv2.imwrite(f'./output/opencv/new/output{i}.jpeg', output)